DAY 7 GOALS

    Create a clean inference pipeline

    Load saved classical ML / DNN / CNN / LSTM / GRU models

    Build a FastAPI DDoS Detection Service

    Add /predict endpoint

    Export preprocessing (scaler + helper script)

    Provide optional Dockerfile (without building it)

In [ ]:
# preprocessing
import numpy as np
import joblib

# Load saved scaler
scaler = joblib.load("scaler.pkl")

def preprocess_sample(data: dict):
    """
    data: dictionary of feature_name: value
    returns: scaled numpy array ready for ML model
    """
    # Convert dict → 2D array
    feature_vector = np.array(list(data.values()), dtype=float).reshape(1, -1)

    # Scale
    scaled = scaler.transform(feature_vector)
    return scaled

def preprocess_for_dl(data: dict):
    scaled = preprocess_sample(data)
    # reshape for CNN/LSTM/GRU
    return scaled.reshape((scaled.shape[0], scaled.shape[1], 1))


In [ ]:
# load_models
import joblib
import tensorflow as tf

# Classical ML models
lr_model  = joblib.load("lr_model.pkl")
rf_model  = joblib.load("rf_model.pkl")
xgb_model = joblib.load("xgb_model.pkl")
lgb_model = joblib.load("lgb_model.pkl")

# Deep Models
dnn_model  = tf.keras.models.load_model("ddos_dnn_model.h5")
cnn_model  = tf.keras.models.load_model("cnn_ddos_model.h5")
lstm_model = tf.keras.models.load_model("lstm_ddos_model.h5")
gru_model  = tf.keras.models.load_model("gru_ddos_model.h5")


In [ ]:
# schemas
from pydantic import BaseModel

class FlowInput(BaseModel):
    # Add all final feature names from CICDDoS2019 after cleaning
    # Example (you must fill in full list):
    Feature1: float
    Feature2: float
    Feature3: float
    # ... will be filled after semester exams
    # FeatureN: float


Build the FastAPI service

In [ ]:
from fastapi import FastAPI
from schemas import FlowInput
from preprocessing import preprocess_sample, preprocess_for_dl
from load_models import (
    lr_model, rf_model, xgb_model, lgb_model,
    dnn_model, cnn_model, lstm_model, gru_model
)

app = FastAPI(title="DDoS Detection API")

@app.get("/")
def home():
    return {"message": "DDoS Detection API Running"}

@app.post("/predict")
def predict_ddos(data: FlowInput):
    data_dict = data.dict()

    # classical models
    X = preprocess_sample(data_dict)
    
    lr_pred  = int(lr_model.predict(X)[0])
    rf_pred  = int(rf_model.predict(X)[0])
    xgb_pred = int(xgb_model.predict(X)[0])
    lgb_pred = int(lgb_model.predict(X)[0])

    # deep learning models
    X_dl = preprocess_for_dl(data_dict)

    dnn_pred  = int((dnn_model.predict(X)[0]  > 0.5))
    cnn_pred  = int((cnn_model.predict(X_dl)[0] > 0.5))
    lstm_pred = int((lstm_model.predict(X_dl)[0] > 0.5))
    gru_pred  = int((gru_model.predict(X_dl)[0] > 0.5))

    return {
        "LogisticRegression": lr_pred,
        "RandomForest":       rf_pred,
        "XGBoost":            xgb_pred,
        "LightGBM":           lgb_pred,
        "DNN":                dnn_pred,
        "CNN":                cnn_pred,
        "LSTM":               lstm_pred,
        "GRU":                gru_pred
    }


# **Day 7 – Evaluation, Results & Final Report**
This notebook includes:
- Model Evaluation Metrics
- Confusion Matrix & Classification Report
- ROC Curve
- Feature Importance (for ML models)
- Final Notes

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, classification_report, roc_curve, auc
import seaborn as sns
import joblib

# Load saved models and datasets
X_test = joblib.load('X_test.pkl')
y_test = joblib.load('y_test.pkl')
model = joblib.load('best_model.pkl')

y_pred = model.predict(X_test)
print('Classification Report:')
print(classification_report(y_test, y_pred))